# PPE Detector: YOLOv8s training (Google Colab, T4 GPU)

Fine-tunes YOLOv8s on a construction-site safety dataset to detect hardhats, safety vests,
and missing PPE (`No-Hardhat`, `No-Safety-Vest`).

**Result (test split):** mAP50 = 0.806, mAP50-95 = 0.553. 40 epochs, about 1.1 hours on a T4.

**Dataset:** "construction site safety" by 007 on Roboflow Universe (v3, YOLOv8 export), CC BY 4.0:
https://universe.roboflow.com/007-kd2zw/construction-site-safety-rmep1

In [1]:
!nvidia-smi

Mon Nov 16 06:17:31 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   57C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
!rm -rf /content/ppe /content/*.zip
!mkdir -p /content/ppe
!cp "/content/drive/MyDrive/ppe/construction site safety.v3i.yolov8.zip" /content/ppe-dataset.zip
!unzip -q /content/ppe-dataset.zip -d /content/ppe
!ls /content/ppe

data.yaml  README.dataset.txt  README.roboflow.txt  test  train  valid


In [7]:
!pip install -q ultralytics
import yaml

NAMES = {0: "Mask", 1: "Safety-Vest", 2: "No-Safety-Vest", 3: "Person", 4: "machinery",
         5: "Safety-Cone", 6: "vehicle", 7: "Hardhat", 8: "No-Mask", 9: "No-Hardhat"}
cfg = {"path": "/content/ppe", "train": "train/images", "val": "valid/images",
       "test": "test/images", "nc": 10, "names": NAMES}
yaml.safe_dump(cfg, open("/content/ppe/data_colab.yaml", "w"), sort_keys=False)
print(open("/content/ppe/data_colab.yaml").read())

path: /content/ppe
train: train/images
val: valid/images
test: test/images
nc: 10
names:
  0: Mask
  1: Safety-Vest
  2: No-Safety-Vest
  3: Person
  4: machinery
  5: Safety-Cone
  6: vehicle
  7: Hardhat
  8: No-Mask
  9: No-Hardhat



In [8]:
import glob
for s in ["train", "valid", "test"]:
    print(s, len(glob.glob(f"/content/ppe/{s}/images/*")), len(glob.glob(f"/content/ppe/{s}/labels/*.txt")))

train 5518 5518
valid 555 555
test 278 278


In [10]:
from ultralytics import YOLO

model = YOLO("yolov8s.pt")
results = model.train(data="/content/ppe/data_colab.yaml", epochs=40, imgsz=640, batch=16,
                      patience=10, device=0, workers=2,
                      project="/content/drive/MyDrive/ppe/runs", name="yolov8s_ppe",
                      exist_ok=True, plots=True)

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/ppe/data_colab.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=40, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov8s_ppe, nbs=64, nms=None, opset=None, optimize=False, o

In [11]:
from ultralytics import YOLO
best = YOLO("/content/drive/MyDrive/ppe/runs/yolov8s_ppe/weights/best.pt")
m = best.val(data="/content/ppe/data_colab.yaml", split="test", imgsz=640)
print("mAP50:", round(m.box.map50, 3), " mAP50-95:", round(m.box.map, 3))
for i, c in enumerate(m.box.ap_class_index):
    print(f"{best.names[c]:15s} mAP50={m.box.ap50[i]:.3f}")

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 72 layers, 11,129,454 parameters, 0 gradients, 28.5 GFLOPs
WARNING ⚠️ val: Slow image access detected (ping: 0.0±0.0 ms, read: 16.0±8.9 MB/s, size: 55.3 KB). Use local storage instead of remote/mounted storage for better performance. See https://docs.ultralytics.com/guides/model-training-tips
val: Scanning /content/ppe/test/labels... 278 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 278/278 712.0it/s 0.4s
val: New cache created: /content/ppe/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 18/18 2.9it/s 6.3s
                   all        278       4046      0.912      0.719      0.806      0.553
                  Mask        124        206      0.928      0.752      0.862      0.602
           Safety-Vest        143        343      0.957      0.752      0.839      0.601
        No-Safety-Vest       

In [ ]:
from google.colab import files
base = "/content/drive/MyDrive/ppe/runs/yolov8s_ppe"
files.download(f"{base}/weights/best.pt")
files.download(f"{base}/results.png")
files.download(f"{base}/confusion_matrix_normalized.png")
files.download(f"{base}/BoxPR_curve.png")